# 23CSE301 ML Capstone - Review 2
**Dataset:** Breast Cancer Wisconsin (Diagnostic) - UCI ML Repository (569 samples, 30 numeric features, binary target: malignant / benign)

**Scope:** Classification (all 10 algorithms, consolidated table) + Clustering (K-Means, Agglomerative)

Source: https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

In [1]:
import os, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve,
                             silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, BaggingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage

RS = 42
np.random.seed(RS)
sns.set_theme(style="whitegrid", palette="colorblind")
os.makedirs("data", exist_ok=True); os.makedirs("models", exist_ok=True); os.makedirs("outputs", exist_ok=True)

## 1. Load data and audit
The CSV is also saved to `data/` so the repo contains the raw dataset (required by the repo structure).

In [2]:
bc = load_breast_cancer(as_frame=True)
df = bc.frame.copy()                      # 'target': 0 = malignant, 1 = benign
df.to_csv("data/breast_cancer_wisconsin.csv", index=False)

print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()), "| Duplicates:", int(df.duplicated().sum()))
print("\nClass distribution:\n", df["target"].map({0: "malignant", 1: "benign"}).value_counts())
df.dtypes.value_counts()

Shape: (569, 31)
Missing values: 0 | Duplicates: 0

Class distribution:
 target
benign       357
malignant    212
Name: count, dtype: int64


float64    30
int64       1
Name: count, dtype: int64

## 2. Quick EDA (target balance + correlation)
Observation: classes are mildly imbalanced (~63% benign), so a **stratified** split and weighted F1 are used. Many "mean/worst" features are highly correlated (radius, perimeter, area), which is why PCA works well later.

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
df["target"].map({0: "malignant", 1: "benign"}).value_counts().plot.bar(ax=ax[0], color=sns.color_palette("colorblind")[:2])
ax[0].set_title("Target distribution"); ax[0].set_xlabel("Class"); ax[0].set_ylabel("Count")
sns.heatmap(df.iloc[:, :10].corr(), cmap="coolwarm", ax=ax[1], cbar_kws={"label": "Pearson r"})
ax[1].set_title("Correlation heatmap (mean features)")
plt.tight_layout(); plt.show()

## 3. Split and scale (scaler fitted on TRAIN only - no leakage)

In [4]:
X, y = df.drop(columns="target"), df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)

scaler = StandardScaler().fit(X_train)
Xtr = pd.DataFrame(scaler.transform(X_train), columns=X.columns, index=X_train.index)
Xte = pd.DataFrame(scaler.transform(X_test),  columns=X.columns, index=X_test.index)
print(Xtr.shape, Xte.shape)

(455, 30) (114, 30)


## 4. Train all 10 classifiers (Part A + Part B) on the same split

In [5]:
models = {
    # Part A
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RS),
    "KNN":                 KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes":         GaussianNB(),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, random_state=RS),
    "SVM (SVC)":           SVC(kernel="rbf", C=1, probability=True, random_state=RS),
    # Part B
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=RS),
    "AdaBoost":            AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=RS),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=RS),
    "Bagging":             BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RS), n_estimators=50, random_state=RS),
    "MLP":                 MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", max_iter=1000, random_state=RS),
}

def evaluate(name, model):
    model.fit(Xtr, y_train)
    pred, proba = model.predict(Xte), model.predict_proba(Xte)[:, 1]
    return {"Model": name,
            "Accuracy":  accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, average="weighted"),
            "Recall":    recall_score(y_test, pred, average="weighted"),
            "F1":        f1_score(y_test, pred, average="weighted"),
            "ROC-AUC":   roc_auc_score(y_test, proba)}

results = pd.DataFrame([evaluate(n, m) for n, m in models.items()])
results = results.sort_values("F1", ascending=False).reset_index(drop=True)
results.index += 1
results.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
1,Logistic Regression,0.9825,0.9825,0.9825,0.9825,0.9954
2,SVM (SVC),0.9825,0.9825,0.9825,0.9825,0.9950
3,MLP,0.9649,0.9659,0.9649,0.9651,0.9937
4,KNN,0.9561,0.9561,0.9561,0.9560,0.9788
5,Random Forest,0.9561,0.9561,0.9561,0.9560,0.9932
6,AdaBoost,0.9561,0.9569,0.9561,0.9558,0.9835
7,Gradient Boosting,0.9561,0.9569,0.9561,0.9558,0.9907
8,Bagging,0.9386,0.9384,0.9386,0.9384,0.9921
9,Naive Bayes,0.9298,0.9298,0.9298,0.9298,0.9868
10,Decision Tree,0.9211,0.9234,0.9211,0.9216,0.9163


## 5. Confusion matrices and ROC curves

In [6]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for ax, (name, m) in zip(axes.ravel(), models.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, m.predict(Xte)),
                           display_labels=["malignant", "benign"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
plt.suptitle("Confusion matrices - all 10 classifiers", y=1.02); plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 6))
for name, m in models.items():
    fpr, tpr, _ = roc_curve(y_test, m.predict_proba(Xte)[:, 1])
    plt.plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_test, m.predict_proba(Xte)[:,1]):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Chance")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("ROC curves - all 10 classifiers")
plt.legend(fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()

## 6. Cross-validation (top 2 models) and hyperparameter tuning
Note: for CV, a Pipeline is used so the scaler is re-fitted inside every fold (no leakage).

In [7]:
from sklearn.pipeline import make_pipeline
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
top2 = results["Model"].head(2).tolist()
for n in top2:
    s = cross_val_score(make_pipeline(StandardScaler(), models[n]), X_train, y_train, cv=cv, scoring="f1_weighted")
    print(f"{n:22s} 5-fold CV F1: {s.mean():.4f} +/- {s.std():.4f}")

Logistic Regression    5-fold CV F1: 0.9780 +/- 0.0099
SVM (SVC)              5-fold CV F1: 0.9691 +/- 0.0148


In [8]:
param_grids = {
    "SVM (SVC)":         {"svc__C": [0.1, 1, 10, 100], "svc__kernel": ["rbf", "linear"], "svc__gamma": ["scale", 0.01, 0.001]},
    "Gradient Boosting": {"gradientboostingclassifier__n_estimators": [100, 200],
                          "gradientboostingclassifier__learning_rate": [0.01, 0.05, 0.1],
                          "gradientboostingclassifier__max_depth": [2, 3]},
    "Random Forest":     {"randomforestclassifier__n_estimators": [100, 300],
                          "randomforestclassifier__max_depth": [None, 5, 10]},
    "MLP":               {"mlpclassifier__hidden_layer_sizes": [(32,), (64, 32)],
                          "mlpclassifier__alpha": [1e-4, 1e-2]},
    "Logistic Regression": {"logisticregression__C": [0.01, 0.1, 1, 10]},
}
tune_rows, best_est = [], {}
for name in param_grids:
    base = make_pipeline(StandardScaler(), models[name].__class__(**models[name].get_params()))
    gs = GridSearchCV(base, param_grids[name], cv=cv, scoring="f1_weighted", n_jobs=-1).fit(X_train, y_train)
    pred = gs.predict(X_test)
    default_cv = cross_val_score(make_pipeline(StandardScaler(), models[name].__class__(**models[name].get_params())),
                                 X_train, y_train, cv=cv, scoring="f1_weighted").mean()
    tune_rows.append({"Model": name, "Best params": {k.split('__')[1]: v for k, v in gs.best_params_.items()},
                      "CV F1 (default)": default_cv, "CV F1 (tuned)": gs.best_score_,
                      "Test F1 (default)": results.set_index("Model").loc[name, "F1"],
                      "Test F1 (tuned)": f1_score(y_test, pred, average="weighted")})
    best_est[name] = gs.best_estimator_
tuning = pd.DataFrame(tune_rows)
tuning["CV improvement"] = tuning["CV F1 (tuned)"] - tuning["CV F1 (default)"]   # tuning is judged on CV, not the 114-sample test set
tuning.round(4)

,Model,Best params,CV F1 (default),CV F1 (tuned),Test F1 (default),Test F1 (tuned),CV improvement
0,SVM (SVC),"{'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}",0.9691,0.9757,0.9825,0.9825,0.0065
1,Gradient Boosting,"{'learning_rate': 0.1, 'max_depth': 2, 'n_esti...",0.9516,0.9670,0.9558,0.9471,0.0154
2,Random Forest,"{'max_depth': None, 'n_estimators': 300}",0.9627,0.9649,0.9560,0.9474,0.0022
3,MLP,"{'alpha': 0.0001, 'hidden_layer_sizes': (32,)}",0.9691,0.9780,0.9651,0.9564,0.0089
4,Logistic Regression,{'C': 0.1},0.9780,0.9823,0.9825,0.9736,0.0044


## 7. Final model selection
Selected by tuned **test F1**, with ROC-AUC as tiebreaker. On a test set of only 114 samples, differences of 1 sample are noise - justify with CV score as well.

In [9]:
final_scores = []
for n, est in best_est.items():
    p = est.predict(X_test); pr = est.predict_proba(X_test)[:, 1]
    final_scores.append((n, f1_score(y_test, p, average="weighted"), roc_auc_score(y_test, pr)))
final_scores = pd.DataFrame(final_scores, columns=["Model", "F1", "ROC-AUC"]).sort_values(["F1", "ROC-AUC"], ascending=False)
display(final_scores.round(4))
best_name = final_scores.iloc[0]["Model"]
print("Final model:", best_name)
joblib.dump(best_est[best_name], "models/final_classifier.pkl")

,Model,F1,ROC-AUC
0,SVM (SVC),0.9825,0.9977
4,Logistic Regression,0.9736,0.9957
3,MLP,0.9564,0.9934
2,Random Forest,0.9474,0.9937
1,Gradient Boosting,0.9471,0.9914


Final model: SVM (SVC)


['models/final_classifier.pkl']

## 8. Feature importance (Random Forest)

In [10]:
imp = pd.Series(models["Random Forest"].feature_importances_, index=X.columns).nlargest(10).sort_values()
imp.plot.barh(figsize=(8, 5), color=sns.color_palette("colorblind")[0])
plt.title("Top-10 feature importances (Random Forest)"); plt.xlabel("Importance"); plt.tight_layout(); plt.show()

# CLUSTERING
Labels are NOT used for fitting. They are used only afterwards (Adjusted Rand Index) to sanity-check the clusters.
Features are standardised (distance-based algorithms need this).

In [11]:
Xs = StandardScaler().fit_transform(X)

# Elbow + silhouette for K-Means
ks = range(2, 11)
inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RS).fit(Xs)
    inertias.append(km.inertia_); sils.append(silhouette_score(Xs, km.labels_))
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(ks, inertias, "o-"); ax[0].set_title("K-Means elbow curve"); ax[0].set_xlabel("k"); ax[0].set_ylabel("Inertia")
ax[1].plot(ks, sils, "o-", color="C1"); ax[1].set_title("Silhouette score vs k"); ax[1].set_xlabel("k"); ax[1].set_ylabel("Silhouette")
plt.tight_layout(); plt.show()

Elbow is not sharp; silhouette peaks at k=2, which also matches the two clinical classes. **k = 2** is used.

In [12]:
K = 2
Z = linkage(Xs, method="ward")
plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, show_contracted=True)
plt.title("Agglomerative clustering dendrogram (Ward, last 30 merges)")
plt.xlabel("Cluster (size in brackets)"); plt.ylabel("Distance"); plt.axhline(Z[-2, 2] - 1, color="r", ls="--", label="cut for k=2")
plt.legend(); plt.tight_layout(); plt.show()

In [13]:
def cluster_metrics(name, labels):
    return {"Algorithm": name, "Silhouette": silhouette_score(Xs, labels),
            "Davies-Bouldin": davies_bouldin_score(Xs, labels),
            "Calinski-Harabasz": calinski_harabasz_score(Xs, labels),
            "ARI vs true labels (post-hoc)": adjusted_rand_score(y, labels)}

fits = {"K-Means (k=2)": KMeans(n_clusters=K, n_init=10, random_state=RS).fit_predict(Xs)}
for link in ["ward", "complete", "average", "single"]:
    fits[f"Agglomerative ({link})"] = AgglomerativeClustering(n_clusters=K, linkage=link).fit_predict(Xs)
clus = pd.DataFrame([cluster_metrics(n, l) for n, l in fits.items()]).set_index("Algorithm")
display(clus.round(4))
print("\nCluster sizes:")
for n, l in fits.items(): print(f"{n:26s}", np.bincount(l))

,Silhouette,Davies-Bouldin,Calinski-Harabasz,ARI vs true labels (post-hoc)
Algorithm,,,,
K-Means (k=2),0.3434,1.3205,267.6917,0.6536
Agglomerative (ward),0.3394,1.3700,248.6288,0.5750
Agglomerative (complete),0.6607,0.4497,27.8727,0.0048
Agglomerative (average),0.6340,0.6799,32.4822,0.0073
Agglomerative (single),0.6607,0.4497,27.8727,0.0048



Cluster sizes:
K-Means (k=2)              [375 194]
Agglomerative (ward)       [184 385]
Agglomerative (complete)   [567   2]
Agglomerative (average)    [566   3]
Agglomerative (single)     [567   2]


Silhouette / Calinski-Harabasz: higher is better. Davies-Bouldin: lower is better.

**Do not pick a winner on silhouette alone.** `complete`, `average` and `single` linkage score a high silhouette / low Davies-Bouldin only because they split off a tiny outlier cluster and lump ~everything else together (check the cluster sizes above; ARI ~ 0 confirms they ignore the real structure). **K-Means and Ward** give two meaningful, balanced clusters, the highest Calinski-Harabasz, and far higher ARI. Use these as the main results and mention the linkage degeneracy in the viva.

## PCA (2D) visualisation for every algorithm, and t-SNE for K-Means

In [14]:
P = PCA(n_components=2, random_state=RS).fit(Xs)
Xp = P.transform(Xs)
fig, axes = plt.subplots(1, len(fits), figsize=(5 * len(fits), 4.5))
for ax, (n, l) in zip(axes, fits.items()):
    sc = ax.scatter(Xp[:, 0], Xp[:, 1], c=l, cmap="tab10", s=18, alpha=0.8)
    ax.set_title(n); ax.set_xlabel(f"PC1 ({P.explained_variance_ratio_[0]:.0%})"); ax.set_ylabel(f"PC2 ({P.explained_variance_ratio_[1]:.0%})")
    ax.legend(*sc.legend_elements(), title="Cluster", loc="best")
plt.tight_layout(); plt.show()

In [15]:
Xt = TSNE(n_components=2, perplexity=30, random_state=RS, init="pca").fit_transform(Xs)
plt.figure(figsize=(6.5, 5))
sc = plt.scatter(Xt[:, 0], Xt[:, 1], c=fits["K-Means (k=2)"], cmap="tab10", s=18)
plt.title("t-SNE - K-Means clusters"); plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
plt.legend(*sc.legend_elements(), title="Cluster"); plt.tight_layout(); plt.show()

## Save outputs for the README

In [16]:
results.round(4).to_csv("outputs/classification_comparison.csv", index_label="Rank")
clus.round(4).to_csv("outputs/clustering_metrics.csv")
print("Saved. Now run:  pip freeze | findstr /i \"scikit pandas numpy matplotlib seaborn scipy joblib\" > requirements.txt")

Saved. Now run:  pip freeze | findstr /i "scikit pandas numpy matplotlib seaborn scipy joblib" > requirements.txt
